In [1]:
import os
import numpy as np
import pandas as pd

DATA = "../data" if os.path.exists("../data") else "data"

def load(name):
    for root, _, files in os.walk(DATA):
        for f in files:
            if f.upper().startswith(name) and f.upper().endswith(".XPT"):
                df = pd.read_sas(os.path.join(root, f))
                num = df.select_dtypes("number").columns
                df[num] = df[num].mask(df[num].abs() < 1e-10, 0)
                print("Loaded", f, df.shape)
                return df
    print("NOT FOUND:", name)
    return None

demo = load("DEMO_L")
cbc = load("CBC_L")
ghb = load("GHB_L")
diq = load("DIQ_L")

df = (demo
      .merge(cbc, on="SEQN")
      .merge(ghb.drop(columns="WTPH2YR"), on="SEQN")
      .merge(diq[["SEQN", "DIQ010"]], on="SEQN", how="left"))

coh = df[(df["RIDAGEYR"] >= 18)
         & (df["WTPH2YR"] > 0)
         & df["LBXGH"].notna()
         & ~df["DIQ010"].isin([1, 7, 9])
         & (df["RIDEXPRG"] != 1)].copy()

print("Cohort:", len(coh))
print(sorted(f for _, _, fs in os.walk(DATA) for f in fs if f.upper().endswith(".XPT")))

Loaded DEMO_L.xpt (11933, 27)
Loaded CBC_L.xpt (8727, 23)
Loaded GHB_L.xpt (7199, 3)
Loaded DIQ_L.xpt (11744, 9)
Cohort: 5144
['AGP_L.xpt', 'ALB_CR_L.xpt', 'BCHE_L.xpt', 'BIOPRO_L.xpt', 'CBC_L.xpt', 'CBC_L.xpt', 'COT_L.xpt', 'DEMO_L.xpt', 'DIQ_L.xpt', 'FAR_L.xpt', 'FASTQX_L.xpt', 'FERTIN_L.xpt', 'FFMR_L.xpt', 'FOLATE_L.xpt', 'FOLFMS_L.xpt', 'GHB_L.xpt', 'GHB_L.xpt', 'GLU_L.xpt', 'GLU_L.xpt', 'HDL_L.xpt', 'HEPA_L.xpt', 'HEPBD_L.xpt', 'HEPB_S_L.xpt', 'HEPC_L.xpt', 'HEPE_L.xpt', 'HSCRP_L.xpt', 'IHGEM_L.xpt', 'INS_L.xpt', 'PBCD_L.xpt', 'PFAS_L.xpt', 'TCHOL_L.xpt', 'TFR_L.xpt', 'TRIGLY_L.xpt', 'TST_L.xpt', 'UCPREG_L.xpt', 'VID_L.xpt', 'VOCWB_L.xpt']


In [2]:
files = sorted(f for _, _, fs in os.walk(DATA) for f in fs if f.upper().endswith(".XPT"))
for f in files:
    print(f)

for name in ["BMX_L", "FERTIN_L", "TFR_L", "HSCRP_L"]:
    print(name, "available" if any(f.upper().startswith(name) for f in files) else "NOT downloaded")

AGP_L.xpt
ALB_CR_L.xpt
BCHE_L.xpt
BIOPRO_L.xpt
CBC_L.xpt
CBC_L.xpt
COT_L.xpt
DEMO_L.xpt
DIQ_L.xpt
FAR_L.xpt
FASTQX_L.xpt
FERTIN_L.xpt
FFMR_L.xpt
FOLATE_L.xpt
FOLFMS_L.xpt
GHB_L.xpt
GHB_L.xpt
GLU_L.xpt
GLU_L.xpt
HDL_L.xpt
HEPA_L.xpt
HEPBD_L.xpt
HEPB_S_L.xpt
HEPC_L.xpt
HEPE_L.xpt
HSCRP_L.xpt
IHGEM_L.xpt
INS_L.xpt
PBCD_L.xpt
PFAS_L.xpt
TCHOL_L.xpt
TFR_L.xpt
TRIGLY_L.xpt
TST_L.xpt
UCPREG_L.xpt
VID_L.xpt
VOCWB_L.xpt
BMX_L NOT downloaded
FERTIN_L available
TFR_L available
HSCRP_L available


In [3]:
import urllib.request

url = "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/BMX_L.xpt"
folder = os.path.join(DATA, "raw", "examination")
os.makedirs(folder, exist_ok=True)
path = os.path.join(folder, "BMX_L.xpt")

urllib.request.urlretrieve(url, path)
print("Saved to", path)

bmx = load("BMX_L")
print(bmx[["SEQN", "BMXBMI", "BMXWAIST"]].describe())

Saved to ../data\raw\examination\BMX_L.xpt
Loaded BMX_L.xpt (8860, 22)
                SEQN       BMXBMI     BMXWAIST
count    8860.000000  8471.000000  8190.000000
mean   136345.828217    27.246665    92.120830
std      3453.776310     8.137781    22.050484
min    130378.000000    11.100000    39.800000
25%    133319.750000    21.600000    77.500000
50%    136377.500000    26.400000    92.700000
75%    139336.250000    31.700000   107.000000
max    142310.000000    74.800000   187.000000


In [4]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score

data = coh.merge(bmx[["SEQN", "BMXBMI", "BMXWAIST"]], on="SEQN", how="left")
data["male"] = (data["RIAGENDR"] == 1).astype(int)
y = (data["LBXGH"] >= 5.7).astype(int)

demo_cols = ["RIDAGEYR", "male"]
cbc_cols = [c for c in cbc.columns if c not in ("SEQN", "WTPH2YR")]
print("CBC features:", len(cbc_cols), "| Missing BMI:", data["BMXBMI"].isna().sum())

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
model = make_pipeline(SimpleImputer(strategy="median"),
                      StandardScaler(),
                      LogisticRegression(max_iter=2000))

def score(cols):
    s = cross_val_score(model, data[cols], y, cv=cv, scoring="roc_auc")
    return round(s.mean(), 3), round(s.std(), 3)

sets = {
    "A: age + sex": demo_cols,
    "B: age + sex + CBC": demo_cols + cbc_cols,
    "C: B + BMI + waist": demo_cols + cbc_cols + ["BMXBMI", "BMXWAIST"],
}

rows = []
for name, cols in sets.items():
    m, sd = score(cols)
    rows.append({"Model": name, "Features": len(cols), "AUROC (mean)": m, "AUROC (SD)": sd})
pd.DataFrame(rows)

CBC features: 21 | Missing BMI: 66


,Model,Features,AUROC (mean),AUROC (SD)
0,A: age + sex,2,0.697,0.012
1,B: age + sex + CBC,23,0.770,0.018
2,C: B + BMI + waist,25,0.780,0.014


In [5]:
fer = load("FERTIN_L")
tfr = load("TFR_L")
crp = load("HSCRP_L")

for name, d in [("FERTIN", fer), ("TFR", tfr), ("HSCRP", crp)]:
    cols = [c for c in d.columns if c.startswith("LB")]
    merged = coh[["SEQN"]].merge(d, on="SEQN", how="left")
    have = merged[cols[0]].notna().sum()
    print(f"{name}: columns {cols} | in cohort with a value: {have} of {len(coh)}")

Loaded FERTIN_L.xpt (2564, 4)
Loaded TFR_L.xpt (2564, 4)
Loaded HSCRP_L.xpt (8727, 4)
FERTIN: columns ['LBXFER', 'LBDFERSI'] | in cohort with a value: 1240 of 5144
TFR: columns ['LBXTFR', 'LBDTFRSI'] | in cohort with a value: 1240 of 5144
HSCRP: columns ['LBXHSCRP', 'LBDHRPLC'] | in cohort with a value: 4990 of 5144


In [6]:
data = (data
        .merge(crp[["SEQN", "LBXHSCRP"]], on="SEQN", how="left")
        .merge(fer[["SEQN", "LBXFER"]], on="SEQN", how="left")
        .merge(tfr[["SEQN", "LBXTFR"]], on="SEQN", how="left"))
data["log_crp"] = np.log1p(data["LBXHSCRP"])
y = (data["LBXGH"] >= 5.7).astype(int)

m, sd = score(demo_cols + cbc_cols + ["log_crp"])
print(f"D: B + hs-CRP   AUROC {m}  (SD {sd})   compare B = 0.770")

sub = data[data["LBXFER"].notna()]
print("Iron subgroup:", len(sub),
      "| sex (1=male, 2=female):", sub["RIAGENDR"].value_counts().to_dict(),
      "| age range:", int(sub["RIDAGEYR"].min()), "-", int(sub["RIDAGEYR"].max()),
      "| with HbA1c >= 5.7%:", int((sub["LBXGH"] >= 5.7).sum()))

D: B + hs-CRP   AUROC 0.781  (SD 0.018)   compare B = 0.770
Iron subgroup: 1240 | sex (1=male, 2=female): {2.0: 1240} | age range: 18 - 49 | with HbA1c >= 5.7%: 181


In [7]:
def score_on(d, cols):
    yy = (d["LBXGH"] >= 5.7).astype(int)
    s = cross_val_score(model, d[cols], yy, cv=cv, scoring="roc_auc")
    return round(s.mean(), 3), round(s.std(), 3)

# 1. Exclude anemic participants (WHO: Hb <13 g/dL men, <12 g/dL women)
data["anemic"] = (((data["male"] == 1) & (data["LBXHGB"] < 13))
                  | ((data["male"] == 0) & (data["LBXHGB"] < 12)))
na = data[~data["anemic"]]
print("Anemic excluded:", int(data["anemic"].sum()), "| remaining:", len(na))
print("  Non-anemic  A: age+sex      ", score_on(na, demo_cols))
print("  Non-anemic  B: age+sex+CBC  ", score_on(na, demo_cols + cbc_cols))

# 2. CBC without red-cell features (white cells and platelets only)
rbc = ["LBXRBCSI", "LBXHGB", "LBXHCT", "LBXMCVSI", "LBXMCHSI", "LBXMC", "LBXRDW"]
no_rbc = [c for c in cbc_cols if c not in rbc]
print("Whole cohort  B without red-cell features:", score_on(data, demo_cols + no_rbc))

# 3. Iron subgroup (women 18-49): does ferritin/TfR add to CBC?
sub = data[data["LBXFER"].notna()].copy()
sub["log_fer"] = np.log1p(sub["LBXFER"])
print("Iron subgroup cases (HbA1c >= 5.7%):", int((sub["LBXGH"] >= 5.7).sum()))
print("  Subgroup  B: age+sex+CBC        ", score_on(sub, demo_cols + cbc_cols))
print("  Subgroup  E: B + ferritin + TfR ", score_on(sub, demo_cols + cbc_cols + ["log_fer", "LBXTFR"]))

Anemic excluded: 473 | remaining: 4671
  Non-anemic  A: age+sex       (np.float64(0.703), np.float64(0.019))
  Non-anemic  B: age+sex+CBC   (np.float64(0.779), np.float64(0.017))
Whole cohort  B without red-cell features: (np.float64(0.73), np.float64(0.014))
Iron subgroup cases (HbA1c >= 5.7%): 181
  Subgroup  B: age+sex+CBC         (np.float64(0.766), np.float64(0.021))
  Subgroup  E: B + ferritin + TfR  (np.float64(0.77), np.float64(0.012))


In [8]:
print("Features kept when red-cell measures removed:", no_rbc)

summary = pd.DataFrame([
    ["Whole cohort", "A: age + sex", 0.697, 0.012],
    ["Whole cohort", "B: + CBC", 0.770, 0.018],
    ["Whole cohort", "C: B + BMI + waist", 0.780, 0.014],
    ["Whole cohort", "D: B + hs-CRP", 0.781, 0.018],
    ["Whole cohort", "B without red-cell features", 0.730, 0.014],
    ["Non-anemic (n=4,671)", "A: age + sex", 0.703, 0.019],
    ["Non-anemic (n=4,671)", "B: + CBC", 0.779, 0.017],
    ["Iron subgroup, women 18-49 (n=1,240)", "B: age + sex + CBC", 0.766, 0.021],
    ["Iron subgroup, women 18-49 (n=1,240)", "E: B + ferritin + TfR", 0.770, 0.012],
], columns=["Population", "Model", "AUROC (mean)", "AUROC (SD)"])
summary

Features kept when red-cell measures removed: ['LBXWBCSI', 'LBXLYPCT', 'LBXMOPCT', 'LBXNEPCT', 'LBXEOPCT', 'LBXBAPCT', 'LBDLYMNO', 'LBDMONO', 'LBDNENO', 'LBDEONO', 'LBDBANO', 'LBXPLTSI', 'LBXMPSI', 'LBXNRBC']


,Population,Model,AUROC (mean),AUROC (SD)
0,Whole cohort,A: age + sex,0.697,0.012
1,Whole cohort,B: + CBC,0.770,0.018
2,Whole cohort,C: B + BMI + waist,0.780,0.014
3,Whole cohort,D: B + hs-CRP,0.781,0.018
4,Whole cohort,B without red-cell features,0.730,0.014
5,"Non-anemic (n=4,671)",A: age + sex,0.703,0.019
6,"Non-anemic (n=4,671)",B: + CBC,0.779,0.017
7,"Iron subgroup, women 18-49 (n=1,240)",B: age + sex + CBC,0.766,0.021
8,"Iron subgroup, women 18-49 (n=1,240)",E: B + ferritin + TfR,0.770,0.012


In [9]:
for c in no_rbc:
    print(c)

LBXWBCSI
LBXLYPCT
LBXMOPCT
LBXNEPCT
LBXEOPCT
LBXBAPCT
LBDLYMNO
LBDMONO
LBDNENO
LBDEONO
LBDBANO
LBXPLTSI
LBXMPSI
LBXNRBC
